# CSE 842 — Hallucination Detection Baseline

research question: can we catch unsupported claims in AI-generated text by checking it against the original source?

data: RAGTruth news summaries. each row is one LLM summary of an article, labeled 1 if annotators flagged any hallucinated text, 0 if not.

## 1. Load data

In [15]:
import re
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

responses = pd.read_json("response.jsonl", lines=True)  # llm summaries + hallucination labels
sources = pd.read_json("source_info.jsonl", lines=True)  # original articles

## 2. Prepare data

news summaries only, from the official train split (official test is saved for the final paper). we split train 80/20 by article so no article shows up in both train and val.

In [16]:
# attach each summary to its source article and keep news summaries only
df = responses.merge(sources, on="source_id")
df = df[df["task_type"] == "Summary"].copy()
df["article"] = df["source_info"]
df["summary"] = df["response"]

# label: 1 if any hallucinated span was annotated, else 0
df["hallucination"] = df["labels"].apply(len).gt(0).astype(int)

# train/validation split, grouped by article
train_pool = df[df["split"] == "train"]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(train_pool, groups=train_pool["source_id"]))
train = train_pool.iloc[train_idx]
val = train_pool.iloc[val_idx]

print("Train:", len(train), "| Validation:", len(val))
print("Label counts (0 = clean, 1 = hallucinated):", train_pool["hallucination"].value_counts().to_dict())

Train: 3804 | Validation: 954
Label counts (0 = clean, 1 = hallucinated): {0: 3276, 1: 1482}


## 3. Features

5 features per summary:
- similarity: TF-IDF cosine similarity between article and summary
- length ratio: summary length / article length
- novel word rate: % of summary words not in the article
- novel bigram rate: % of summary word pairs not in the article
- novel number rate: % of summary numbers (dates, amounts) not in the article

these are shallow and can't catch real contradictions, that's why it's a baseline.

In [17]:
# learn vocabulary from training text only
vectorizer = TfidfVectorizer(stop_words="english", max_features=10000)
vectorizer.fit(pd.concat([train["article"], train["summary"]]))


# grab words and numbers from text
def words(text):
    return re.findall(r"\w+", text.lower())


def numbers(text):
    return re.findall(r"\d+(?:[.,]\d+)?", text)


# % of summary items not in the article (0 if summary has none)
def novel_rate(summary_items, article_items):
    if not summary_items:
        return 0.0
    article_set = set(article_items)
    return sum(item not in article_set for item in summary_items) / len(summary_items)


def make_features(data):
    # cosine similarity (tfidf vectors are already normalized, so dot product = cosine)
    article_vecs = vectorizer.transform(data["article"])
    summary_vecs = vectorizer.transform(data["summary"])
    similarity = np.asarray(article_vecs.multiply(summary_vecs).sum(axis=1)).ravel()

    # summary length vs article length
    length_ratio = data["summary"].str.len() / (data["article"].str.len() + 1)

    # novel words, word pairs, and numbers
    novel_word, novel_bigram, novel_number = [], [], []
    for article, summary in zip(data["article"], data["summary"]):
        a_words, s_words = words(article), words(summary)
        novel_word.append(novel_rate(s_words, a_words))
        novel_bigram.append(novel_rate(list(zip(s_words, s_words[1:])), list(zip(a_words, a_words[1:]))))
        novel_number.append(novel_rate(numbers(summary), numbers(article)))

    return np.column_stack([similarity, length_ratio, novel_word, novel_bigram, novel_number])


# feature matrices + labels for train and val
X_train, y_train = make_features(train), train["hallucination"]
X_val, y_val = make_features(val), val["hallucination"]

## 4. Models

- majority class: always predicts "no hallucination", just a floor to compare against
- logistic regression: uses the 5 features. `class_weight="balanced"` makes up for having fewer hallucinated examples

In [18]:
models = {
    "Majority class": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
}

# train both models
for model in models.values():
    model.fit(X_train, y_train)

## 5. Validation results

hallucinated = positive class. F1 is the main metric since accuracy is misleading with imbalanced labels.

In [19]:
# score each model on val
rows = []
for name, model in models.items():
    pred = model.predict(X_val)
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, pred),
        "Precision": precision_score(y_val, pred, zero_division=0),
        "Recall": recall_score(y_val, pred, zero_division=0),
        "F1": f1_score(y_val, pred, zero_division=0),
    })

results = pd.DataFrame(rows).round(3)
results

,Model,Accuracy,Precision,Recall,F1
0,Majority class,0.718,0.000,0.000,0.000
1,Logistic Regression,0.585,0.366,0.643,0.466


## 6. Preliminary: DeBERTa zero-shot NLI

an off-the-shelf DeBERTa NLI model checks if the article supports each summary sentence. no fine-tuning yet, that's for the final paper.

if any sentence has entailment prob < 0.5, the summary gets flagged as hallucinated. the 0.5 is fixed, not tuned on val.

limitation: DeBERTa only reads 512 tokens, so long articles get cut off.

In [20]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# load pretrained NLI model (uses mac gpu if available)
nli_name = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
device = "mps" if torch.backends.mps.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(nli_name)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_name).to(device).eval()

# find which model output is "entailment"
entail_id = {label.lower(): i for i, label in nli_model.config.id2label.items()}["entailment"]


# split summary into sentences, skip tiny fragments
def split_sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if len(s.split()) > 2]


# entailment prob for each (article, sentence) pair, in batches
@torch.inference_mode()
def entailment_probs(premises, hypotheses, batch_size=16):
    probs = []
    for i in range(0, len(premises), batch_size):
        batch = tokenizer(
            premises[i:i + batch_size],
            hypotheses[i:i + batch_size],
            truncation="only_first",  # cut the article, never the sentence
            max_length=512,
            padding=True,
            return_tensors="pt",
        ).to(device)
        logits = nli_model(**batch).logits
        probs.extend(logits.softmax(dim=-1)[:, entail_id].cpu().tolist())
    return np.array(probs)


print("Device:", device)

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 8504.47it/s]


Device: mps


In [21]:
# one (article, sentence) pair per summary sentence
pairs = [
    (row_id, article, sentence)
    for row_id, article, summary in zip(range(len(val)), val["article"], val["summary"])
    for sentence in split_sentences(summary)
]
row_ids, premises, hypotheses = map(list, zip(*pairs))
print("Validation summaries:", len(val), "| sentence pairs:", len(pairs))

# summary score = prob of its least supported sentence, flag if < 0.5
probs = entailment_probs(premises, hypotheses)
min_entail = pd.Series(probs).groupby(row_ids).min().reindex(range(len(val)), fill_value=1.0).to_numpy()
deberta_pred = (min_entail < 0.5).astype(int)

# same metrics as the baselines, then add to the results table
deberta_row = {
    "Model": "DeBERTa zero-shot NLI",
    "Accuracy": accuracy_score(y_val, deberta_pred),
    "Precision": precision_score(y_val, deberta_pred, zero_division=0),
    "Recall": recall_score(y_val, deberta_pred, zero_division=0),
    "F1": f1_score(y_val, deberta_pred, zero_division=0),
}
pd.concat([results, pd.DataFrame([deberta_row]).round(3)], ignore_index=True)

Validation summaries: 954 | sentence pairs: 5181


,Model,Accuracy,Precision,Recall,F1
0,Majority class,0.718,0.000,0.000,0.000
1,Logistic Regression,0.585,0.366,0.643,0.466
2,DeBERTa zero-shot NLI,0.531,0.348,0.758,0.477
